## 1. Understanding the GTFS Dataset

### 1.1 Introduction

General Transit Feed Specification (GTFS) is a standard format used to represent public transportation information. It contains details about routes, trips, stops, schedules and service calendars.

The New York MTA GTFS dataset will be inspected to understand how buses operate and how the different data files are connected.

### 1.2 Important GTFS Files

The following files are important for this project:

| File Name | Description |
|---|---|
| routes.txt | Contains information about transit routes |
| trips.txt | Contains individual scheduled trips |
| stops.txt | Contains bus stop information |
| stop_times.txt | Contains arrival and departure times at stops |
| calendar.txt | Defines regular weekly service |
| calendar_dates.txt | Defines additional or removed service dates |
| shapes.txt | Contains route geographical information |

**Note:** The availability of these files will be verified from the downloaded GTFS feed before further processing.

In [1]:
import requests
import zipfile
from io import BytesIO

# New York MTA GTFS feed
gtfs_url = "https://rrgtfsfeeds.s3.amazonaws.com/gtfs_supplemented.zip"

# Download the GTFS ZIP file
response = requests.get(gtfs_url, timeout=120)

print("Download Status:", response.status_code)
response.raise_for_status()

# Open the ZIP file
gtfs_zip = zipfile.ZipFile(BytesIO(response.content))

# Display available files
gtfs_files = gtfs_zip.namelist()

print("\nAvailable GTFS Files:")
for file in gtfs_files:
    print(file)

Download Status: 200

Available GTFS Files:
feed_info.txt
agency.txt
shapes.txt
routes.txt
stops.txt
trips.txt
stop_times.txt
calendar.txt
calendar_dates.txt
transfers.txt


## 2. Loading and Inspecting GTFS Files

### 2.1 Loading the Dataset

The GTFS feed contains multiple text files, and each file represents a different part of the public transportation system.

In this section, the available files are loaded into Pandas DataFrames. Their dimensions, column names and sample records are examined to understand the dataset structure.

### 2.2 Dataset Structure

The inspection identifies the number of records and columns in each file. This helps us understand the amount and type of information available for further analysis.

In [2]:
import pandas as pd

# Important GTFS files
required_files = [
    "routes.txt",
    "trips.txt",
    "stops.txt",
    "stop_times.txt",
    "calendar.txt",
    "calendar_dates.txt",
    "shapes.txt",
    "transfers.txt"
]

# Load available files into DataFrames
gtfs_data = {}

for file in required_files:
    if file in gtfs_files:
        df_temp = pd.read_csv(
            gtfs_zip.open(file),
            low_memory=False
        )
        gtfs_data[file.replace(".txt", "")] = df_temp

# Display dataset dimensions
dataset_summary = []

for name, data in gtfs_data.items():
    dataset_summary.append({
        "GTFS File": name + ".txt",
        "Number of Rows": data.shape[0],
        "Number of Columns": data.shape[1]
    })

summary_df = pd.DataFrame(dataset_summary)

display(summary_df)

,GTFS File,Number of Rows,Number of Columns
0,routes.txt,29,10
1,trips.txt,72437,6
2,stops.txt,1488,6
3,stop_times.txt,2139686,5
4,calendar.txt,13,10
5,calendar_dates.txt,78,3
6,shapes.txt,153567,4
7,transfers.txt,613,4


## 3. Exploring GTFS File Structure

### 3.1 Column Inspection

Each GTFS file contains different attributes describing the public transportation network. Examining the column names helps identify the information available for analysis.

### 3.2 Sample Record Inspection

A few sample records are displayed from each file to understand how the information is stored and what each record represents.

This step helps identify important fields that will be used during data validation, cleaning and feature engineering.

In [3]:
# Display column names of all GTFS files

print("GTFS COLUMN INFORMATION")
print("=" * 60)

for name, data in gtfs_data.items():
    print(f"\n{name.upper()}.TXT")
    print("-" * 40)
    print("Columns:", list(data.columns))

# Display sample records from important files

important_files = [
    "routes",
    "trips",
    "stops",
    "stop_times",
    "calendar"
]

for name in important_files:
    print(f"\nSAMPLE RECORDS: {name.upper()}")
    display(gtfs_data[name].head(3))

GTFS COLUMN INFORMATION

ROUTES.TXT
----------------------------------------
Columns: ['agency_id', 'route_id', 'route_short_name', 'route_long_name', 'route_type', 'route_desc', 'route_url', 'route_color', 'route_text_color', 'route_sort_order']

TRIPS.TXT
----------------------------------------
Columns: ['route_id', 'trip_id', 'service_id', 'trip_headsign', 'direction_id', 'shape_id']

STOPS.TXT
----------------------------------------
Columns: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'location_type', 'parent_station']

STOP_TIMES.TXT
----------------------------------------
Columns: ['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'stop_sequence']

CALENDAR.TXT
----------------------------------------
Columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

CALENDAR_DATES.TXT
----------------------------------------
Columns: ['service_id', 'date', 'exception_type']

SHAPES.TXT
------------------

,agency_id,route_id,route_short_name,route_long_name,route_type,route_desc,route_url,route_color,route_text_color,route_sort_order
0,MTA NYCT,1,1,Broadway - 7 Avenue Local,1,Trains operate between Van Cortlandt Park-242 ...,https://www.mta.info/schedules/subway/1-train,D82233,FFFFFF,20
1,MTA NYCT,2,2,7 Avenue Express,1,"Trains operate between Wakefield-241 St, Bronx...",https://www.mta.info/schedules/subway/2-train,D82233,FFFFFF,21
2,MTA NYCT,3,3,7 Avenue Express,1,"Trains operate between Harlem-148 St, Manhatta...",https://www.mta.info/schedules/subway/3-train,D82233,FFFFFF,22



SAMPLE RECORDS: TRIPS


,route_id,trip_id,service_id,trip_headsign,direction_id,shape_id
0,1,AFA26GEN-1038-Sunday-00_000600_1..S03R,Sunday_C6,South Ferry,1,1..S03R
1,1,AFA26GEN-1038-Sunday-00_002600_1..S03R,Sunday_C6,South Ferry,1,1..S03R
2,1,AFA26GEN-1038-Sunday-00_004600_1..S03R,Sunday_C6,South Ferry,1,1..S03R



SAMPLE RECORDS: STOPS


,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
0,101,Van Cortlandt Park-242 St,40.889248,-73.898583,1.0,NaN
1,101N,Van Cortlandt Park-242 St,40.889248,-73.898583,NaN,101
2,101S,Van Cortlandt Park-242 St,40.889248,-73.898583,NaN,101



SAMPLE RECORDS: STOP_TIMES


,trip_id,stop_id,arrival_time,departure_time,stop_sequence
0,AFA26GEN-1038-Sunday-00_000600_1..S03R,101S,00:06:00,00:06:00,1
1,AFA26GEN-1038-Sunday-00_000600_1..S03R,103S,00:07:30,00:07:30,2
2,AFA26GEN-1038-Sunday-00_000600_1..S03R,104S,00:09:00,00:09:00,3



SAMPLE RECORDS: CALENDAR


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,Saturday,0,0,0,0,0,1,0,20261101,20270515
1,Saturday_C1,0,0,0,0,0,1,0,20261101,20270515
2,Saturday_C2,0,0,0,0,0,1,0,20261101,20270515


## 4. Data Quality Assessment

### 4.1 Missing Value Analysis

Missing values occur when certain fields do not contain information. Identifying them is important because incomplete records may affect further analysis.

### 4.2 Duplicate Record Analysis

Duplicate records are repeated entries in a dataset. They can affect the accuracy of data analysis and should be identified before cleaning.

In this section, missing values and duplicate records are examined across all available GTFS files.

In [4]:
# Check missing values and duplicate records

quality_results = []

for name, data in gtfs_data.items():

    quality_results.append({
        "GTFS File": name + ".txt",
        "Total Rows": len(data),
        "Missing Values": data.isnull().sum().sum(),
        "Duplicate Rows": data.duplicated().sum()
    })

quality_df = pd.DataFrame(quality_results)

print("DATA QUALITY SUMMARY")
display(quality_df)

# Identify columns containing missing values

print("\nMISSING VALUES BY COLUMN")

for name, data in gtfs_data.items():

    missing = data.isnull().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n{name}.txt")
        display(missing.to_frame("Missing Count"))

DATA QUALITY SUMMARY


,GTFS File,Total Rows,Missing Values,Duplicate Rows
0,routes.txt,29,0,0
1,trips.txt,72437,6958,0
2,stops.txt,1488,1488,0
3,stop_times.txt,2139686,0,0
4,calendar.txt,13,0,0
5,calendar_dates.txt,78,0,0
6,shapes.txt,153567,0,0
7,transfers.txt,613,0,0



MISSING VALUES BY COLUMN

trips.txt


,Missing Count
shape_id,6958



stops.txt


,Missing Count
location_type,992
parent_station,496


## 5. GTFS Data Relationship Validation

### 5.1 Introduction

GTFS files are connected through common identifiers such as route_id, trip_id, stop_id and service_id.

These identifiers help maintain consistency between routes, trips, stops and schedules.

### 5.2 Relationship Checks

The following relationships are examined:

- routes.txt and trips.txt are connected using route_id.
- trips.txt and stop_times.txt are connected using trip_id.
- stops.txt and stop_times.txt are connected using stop_id.
- trips.txt and calendar.txt are connected using service_id.

### 5.3 Objective

The objective is to identify records that refer to missing or invalid identifiers. These checks help determine whether the GTFS dataset is structurally consistent.

In [5]:
# Validate relationships between GTFS files

def check_relationship(parent_df, parent_column,
                       child_df, child_column):

    if parent_column not in parent_df.columns:
        return "Parent column not found"

    if child_column not in child_df.columns:
        return "Child column not found"

    parent_values = set(parent_df[parent_column].dropna())
    child_values = child_df[child_column].dropna()

    unmatched = (~child_values.isin(parent_values)).sum()

    return int(unmatched)


relationships = [
    ("routes", "route_id", "trips", "route_id"),
    ("trips", "trip_id", "stop_times", "trip_id"),
    ("stops", "stop_id", "stop_times", "stop_id"),
    ("calendar", "service_id", "trips", "service_id")
]

relationship_results = []

for parent, parent_col, child, child_col in relationships:

    result = check_relationship(
        gtfs_data[parent], parent_col,
        gtfs_data[child], child_col
    )

    relationship_results.append({
        "Parent File": parent + ".txt",
        "Child File": child + ".txt",
        "Relationship": parent_col + " → " + child_col,
        "Unmatched Records": result
    })

relationship_df = pd.DataFrame(relationship_results)

print("GTFS RELATIONSHIP VALIDATION")
display(relationship_df)

GTFS RELATIONSHIP VALIDATION


,Parent File,Child File,Relationship,Unmatched Records
0,routes.txt,trips.txt,route_id → route_id,0
1,trips.txt,stop_times.txt,trip_id → trip_id,0
2,stops.txt,stop_times.txt,stop_id → stop_id,0
3,calendar.txt,trips.txt,service_id → service_id,52127


## 6. Route Profile and Trip Frequency Analysis

### 6.1 Introduction

A route profile provides basic information about each public transportation route. It helps us understand the available routes and the number of scheduled trips associated with each route.

### 6.2 Trip Frequency Analysis

Trip frequency represents the number of scheduled trips associated with a route in the collected GTFS feed.

In this section, route information is combined with trip information using the common route_id field.

### 6.3 Expected Outcomes

- Identify the total number of routes.
- Calculate the number of scheduled trips for each route.
- Understand route-wise service distribution.
- Prepare basic route-level features for further analysis.

**Note:** Scheduled trip frequency represents planned service, not actual passenger ridership.

In [6]:
# Create route-level profiles using routes and trips

routes_df = gtfs_data["routes"].copy()
trips_df = gtfs_data["trips"].copy()

# Count scheduled trips for each route
trip_frequency = (
    trips_df.groupby("route_id")
    .size()
    .reset_index(name="total_scheduled_trips")
)

# Combine route information with trip frequency
route_profiles = routes_df.merge(
    trip_frequency,
    on="route_id",
    how="left"
)

route_profiles["total_scheduled_trips"] = (
    route_profiles["total_scheduled_trips"].fillna(0).astype(int)
)

# Display route profile
print("TOTAL NUMBER OF ROUTES:", len(route_profiles))

print("\nROUTE PROFILES")
display(route_profiles.head(10))

# Display routes with the highest scheduled trip counts
print("\nTOP 10 ROUTES BY SCHEDULED TRIP COUNT")

top_routes = route_profiles.sort_values(
    by="total_scheduled_trips",
    ascending=False
).head(10)

display(top_routes)

TOTAL NUMBER OF ROUTES: 29

ROUTE PROFILES


,agency_id,route_id,route_short_name,route_long_name,route_type,route_desc,route_url,route_color,route_text_color,route_sort_order,total_scheduled_trips
0,MTA NYCT,1,1,Broadway - 7 Avenue Local,1,Trains operate between Van Cortlandt Park-242 ...,https://www.mta.info/schedules/subway/1-train,D82233,FFFFFF,20,4100
1,MTA NYCT,2,2,7 Avenue Express,1,"Trains operate between Wakefield-241 St, Bronx...",https://www.mta.info/schedules/subway/2-train,D82233,FFFFFF,21,3014
2,MTA NYCT,3,3,7 Avenue Express,1,"Trains operate between Harlem-148 St, Manhatta...",https://www.mta.info/schedules/subway/3-train,D82233,FFFFFF,22,3071
3,MTA NYCT,4,4,Lexington Avenue Express,1,"Trains operate daily between Woodlawn, Bronx, ...",https://www.mta.info/schedules/subway/4-train,009952,FFFFFF,23,3802
4,MTA NYCT,5,5,Lexington Avenue Express,1,"Weekdays daytime, most trains operate between ...",https://www.mta.info/schedules/subway/5-train,009952,FFFFFF,24,2573
5,MTA NYCT,6,6,Lexington Avenue Local,1,"Local trains operate between Pelham Bay Park, ...",https://www.mta.info/schedules/subway/6-train,009952,FFFFFF,25,3952
6,MTA NYCT,6X,6X,Pelham Bay Park Express,1,Express trains operate between Pelham Bay Park...,https://www.mta.info/schedules/subway/6-train,009952,FFFFFF,26,456
7,MTA NYCT,7,7,Flushing Local,1,"Trains operate between Flushing-Main St, Queen...",https://www.mta.info/schedules/subway/7-train,9A38A1,FFFFFF,27,4736
8,MTA NYCT,7X,7X,Flushing Express,1,"Trains operate between Flushing-Main St, Queen...",https://www.mta.info/schedules/subway/7-train,9A38A1,FFFFFF,28,428
9,MTA NYCT,A,A,8 Avenue Express,1,"Trains operate between Inwood-207 St, Manhatta...",https://www.mta.info/schedules/subway/a-train,0062CF,FFFFFF,1,3401



TOP 10 ROUTES BY SCHEDULED TRIP COUNT


,agency_id,route_id,route_short_name,route_long_name,route_type,route_desc,route_url,route_color,route_text_color,route_sort_order,total_scheduled_trips
21,MTA NYCT,L,L,14 St-Canarsie Local,1,"Trains operate between 8 Av, Manhattan, and Ca...",https://www.mta.info/schedules/subway/l-train,7C858C,FFFFFF,12,6048
7,MTA NYCT,7,7,Flushing Local,1,"Trains operate between Flushing-Main St, Queen...",https://www.mta.info/schedules/subway/7-train,9A38A1,FFFFFF,27,4736
0,MTA NYCT,1,1,Broadway - 7 Avenue Local,1,Trains operate between Van Cortlandt Park-242 ...,https://www.mta.info/schedules/subway/1-train,D82233,FFFFFF,20,4100
5,MTA NYCT,6,6,Lexington Avenue Local,1,"Local trains operate between Pelham Bay Park, ...",https://www.mta.info/schedules/subway/6-train,009952,FFFFFF,25,3952
3,MTA NYCT,4,4,Lexington Avenue Express,1,"Trains operate daily between Woodlawn, Bronx, ...",https://www.mta.info/schedules/subway/4-train,009952,FFFFFF,23,3802
13,MTA NYCT,E,E,8 Avenue Local,1,Trains operate between Jamaica Center-Parsons/...,https://www.mta.info/schedules/subway/e-train,0062CF,FFFFFF,3,3514
9,MTA NYCT,A,A,8 Avenue Express,1,"Trains operate between Inwood-207 St, Manhatta...",https://www.mta.info/schedules/subway/a-train,0062CF,FFFFFF,1,3401
14,MTA NYCT,F,F,Queens Blvd Express/6 Av Local,1,Trains operate at all times between Jamaica-17...,https://www.mta.info/schedules/subway/f-train,EB6800,FFFFFF,6,3396
17,MTA NYCT,G,G,Brooklyn-Queens Crosstown,1,"Trains operate between Court Sq, Queens and Ch...",https://www.mta.info/schedules/subway/g-train,799534,FFFFFF,9,3286
24,MTA NYCT,Q,Q,Broadway Express,1,"Trains operate between 96 St (2 Av), Manhattan...",https://www.mta.info/schedules/subway/q-train,F6BC26,000000,14,3177


## 7. Weekday and Weekend Service Analysis

### 7.1 Introduction

Public transportation services may operate differently on weekdays and weekends. Understanding these differences helps identify variations in scheduled bus services.

### 7.2 Service Calendar Analysis

The GTFS calendar.txt file contains weekly service information. It indicates whether a particular service operates on Monday, Tuesday, Wednesday, Thursday, Friday, Saturday and Sunday.

In this section, trip information is combined with calendar information to compare scheduled weekday and weekend trip counts for each route.

### 7.3 Expected Outcomes

- Identify weekday and weekend service patterns.
- Calculate scheduled trips for each route.
- Compare weekday and weekend service levels.
- Prepare service-related features for further analysis.

**Note:** This analysis uses regular calendar schedules. Special service additions and removals in calendar_dates.txt will need to be considered separately for date-specific analysis.

In [7]:
# Weekday and weekend service analysis

calendar_df = gtfs_data["calendar"].copy()
trips_df = gtfs_data["trips"].copy()

# Define weekday and weekend columns
weekday_columns = [
    "monday", "tuesday", "wednesday",
    "thursday", "friday"
]

weekend_columns = ["saturday", "sunday"]

# Combine trips with service calendar
service_trips = trips_df.merge(
    calendar_df,
    on="service_id",
    how="inner"
)

# Count weekday and weekend scheduled trips
weekday_trips = (
    service_trips[
        service_trips[weekday_columns].eq(1).any(axis=1)
    ]
    .groupby("route_id")
    .size()
    .reset_index(name="weekday_scheduled_trips")
)

weekend_trips = (
    service_trips[
        service_trips[weekend_columns].eq(1).any(axis=1)
    ]
    .groupby("route_id")
    .size()
    .reset_index(name="weekend_scheduled_trips")
)

# Combine both results
service_comparison = route_profiles[
    ["route_id", "total_scheduled_trips"]
].merge(
    weekday_trips, on="route_id", how="left"
).merge(
    weekend_trips, on="route_id", how="left"
)

# Replace missing counts with zero
service_comparison[
    ["weekday_scheduled_trips", "weekend_scheduled_trips"]
] = service_comparison[
    ["weekday_scheduled_trips", "weekend_scheduled_trips"]
].fillna(0).astype(int)

print("WEEKDAY AND WEEKEND SERVICE COMPARISON")
display(service_comparison.head(10))

# Overall summary
print("\nOVERALL SERVICE SUMMARY")

display(
    service_comparison[
        [
            "weekday_scheduled_trips",
            "weekend_scheduled_trips"
        ]
    ].sum().to_frame("Total Scheduled Trips")
)

WEEKDAY AND WEEKEND SERVICE COMPARISON


,route_id,total_scheduled_trips,weekday_scheduled_trips,weekend_scheduled_trips
0,1,4100,462,680
1,2,3014,324,524
2,3,3071,302,461
3,4,3802,368,534
4,5,2573,329,472
5,6,3952,438,658
6,6X,456,114,0
7,7,4736,535,812
8,7X,428,107,0
9,A,3401,379,561



OVERALL SERVICE SUMMARY


,Total Scheduled Trips
weekday_scheduled_trips,8498
weekend_scheduled_trips,11812


## 8. Final Route-Level Feature Engineering

### 8.1 Introduction

Feature engineering is the process of creating useful variables from existing data. In this section, route information, scheduled trip frequency and stop coverage are combined to create a route-level dataset.

### 8.2 Features Created

- Route ID: Unique identifier of each route.
- Route Name: Name or short name of the route, if available.
- Total Scheduled Trips: Number of trips assigned to each route.
- Unique Stops: Number of different stops served by each route.
- Average Stops per Trip: Average number of stop-time records per scheduled trip.

### 8.3 Objective

The final route-level dataset will provide basic network and service information for further analysis and route optimization.

In [8]:
# Create route-level features

routes_df = gtfs_data["routes"].copy()
trips_df = gtfs_data["trips"].copy()
stop_times_df = gtfs_data["stop_times"].copy()

# Count scheduled trips per route
trip_counts = (
    trips_df.groupby("route_id")["trip_id"]
    .nunique()
    .reset_index(name="total_scheduled_trips")
)

# Connect trips with stop-time records
trip_stop_data = trips_df[
    ["trip_id", "route_id"]
].merge(
    stop_times_df[["trip_id", "stop_id"]],
    on="trip_id",
    how="left"
)

# Count unique stops per route
stop_counts = (
    trip_stop_data.groupby("route_id")["stop_id"]
    .nunique()
    .reset_index(name="unique_stops")
)

# Count stop-time records per route
stop_time_counts = (
    trip_stop_data.groupby("route_id")
    .size()
    .reset_index(name="total_stop_time_records")
)

# Combine route-level features
final_route_features = (
    routes_df.merge(trip_counts, on="route_id", how="left")
    .merge(stop_counts, on="route_id", how="left")
    .merge(stop_time_counts, on="route_id", how="left")
)

# Calculate average stops per scheduled trip
final_route_features["average_stops_per_trip"] = (
    final_route_features["total_stop_time_records"]
    / final_route_features["total_scheduled_trips"].replace(0, pd.NA)
)

# Display final features
print("FINAL ROUTE-LEVEL DATASET")
display(final_route_features.head(10))

print("\nDataset Shape:", final_route_features.shape)

FINAL ROUTE-LEVEL DATASET


,agency_id,route_id,route_short_name,route_long_name,route_type,route_desc,route_url,route_color,route_text_color,route_sort_order,total_scheduled_trips,unique_stops,total_stop_time_records,average_stops_per_trip
0,MTA NYCT,1,1,Broadway - 7 Avenue Local,1,Trains operate between Van Cortlandt Park-242 ...,https://www.mta.info/schedules/subway/1-train,D82233,FFFFFF,20,4100,76,152362,37.161463
1,MTA NYCT,2,2,7 Avenue Express,1,"Trains operate between Wakefield-241 St, Bronx...",https://www.mta.info/schedules/subway/2-train,D82233,FFFFFF,21,3014,142,152422,50.571334
2,MTA NYCT,3,3,7 Avenue Express,1,"Trains operate between Harlem-148 St, Manhatta...",https://www.mta.info/schedules/subway/3-train,D82233,FFFFFF,22,3071,68,91880,29.918593
3,MTA NYCT,4,4,Lexington Avenue Express,1,"Trains operate daily between Woodlawn, Bronx, ...",https://www.mta.info/schedules/subway/4-train,009952,FFFFFF,23,3802,108,118633,31.202788
4,MTA NYCT,5,5,Lexington Avenue Express,1,"Weekdays daytime, most trains operate between ...",https://www.mta.info/schedules/subway/5-train,009952,FFFFFF,24,2573,101,62877,24.437233
5,MTA NYCT,6,6,Lexington Avenue Local,1,"Local trains operate between Pelham Bay Park, ...",https://www.mta.info/schedules/subway/6-train,009952,FFFFFF,25,3952,76,142718,36.112854
6,MTA NYCT,6X,6X,Pelham Bay Park Express,1,Express trains operate between Pelham Bay Park...,https://www.mta.info/schedules/subway/6-train,009952,FFFFFF,26,456,58,13224,29.000000
7,MTA NYCT,7,7,Flushing Local,1,"Trains operate between Flushing-Main St, Queen...",https://www.mta.info/schedules/subway/7-train,9A38A1,FFFFFF,27,4736,44,98106,20.714949
8,MTA NYCT,7X,7X,Flushing Express,1,"Trains operate between Flushing-Main St, Queen...",https://www.mta.info/schedules/subway/7-train,9A38A1,FFFFFF,28,428,36,7131,16.661215
9,MTA NYCT,A,A,8 Avenue Express,1,"Trains operate between Inwood-207 St, Manhatta...",https://www.mta.info/schedules/subway/a-train,0062CF,FFFFFF,1,3401,131,113810,33.463687



Dataset Shape: (29, 14)


## 9. Final Dataset Validation

### Introduction
This section checks the completeness and consistency of the engineered route-level dataset.

### Validation Checks
- Check missing values in the generated features.
- Identify duplicate route records.
- Verify the number of routes.
- Review the data types of the columns.

### Objective
To ensure that the route-level dataset is suitable for further data processing and exploratory data analysis.

In [9]:
# Validate the final route-level dataset

print("FINAL DATASET VALIDATION")
print("=" * 50)

print("\n1. Dataset Shape:")
print(final_route_features.shape)

print("\n2. Missing Values:")
display(
    final_route_features.isnull()
    .sum()
    .to_frame("Missing Values")
)

print("\n3. Duplicate Route IDs:")
print(final_route_features["route_id"].duplicated().sum())

print("\n4. Column Data Types:")
display(
    final_route_features.dtypes
    .to_frame("Data Type")
)

print("\n5. Summary Statistics:")
display(final_route_features.describe(include="all").T)

FINAL DATASET VALIDATION

1. Dataset Shape:
(29, 14)

2. Missing Values:


,Missing Values
agency_id,0
route_id,0
route_short_name,0
route_long_name,0
route_type,0
route_desc,0
route_url,0
route_color,0
route_text_color,0
route_sort_order,0



3. Duplicate Route IDs:
0

4. Column Data Types:


,Data Type
agency_id,object
route_id,object
route_short_name,object
route_long_name,object
route_type,int64
route_desc,object
route_url,object
route_color,object
route_text_color,object
route_sort_order,int64



5. Summary Statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
agency_id,29,1,MTA NYCT,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_id,29,29,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_short_name,29,27,S,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_long_name,29,23,Broadway Local,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_type,29.0,NaN,NaN,NaN,1.034483,0.185695,1.0,1.0,1.0,1.0,2.0
route_desc,29,29,Trains operate between Van Cortlandt Park-242 ...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_url,29,25,https://www.mta.info/schedules/subway/7-train,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_color,29,10,EB6800,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_text_color,29,2,FFFFFF,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
route_sort_order,29.0,NaN,NaN,NaN,15.0,8.514693,1.0,8.0,15.0,22.0,29.0


## 10. Exporting the Processed Route-Level Dataset

### Introduction
After completing route-level feature engineering and validation, the processed dataset is exported into CSV format.

### Objective
To save the engineered route information so it can be reused in future exploratory data analysis, machine learning and route optimization tasks.

In [10]:
# Export the final route-level dataset

output_file = "new_york_route_features.csv"

final_route_features.to_csv(output_file, index=False)

print("Dataset exported successfully!")
print("File Name:", output_file)
print("Total Routes:", len(final_route_features))
print("Total Features:", len(final_route_features.columns))

Dataset exported successfully!
File Name: new_york_route_features.csv
Total Routes: 29
Total Features: 14
